In [2]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor

In [5]:
df = pd.read_csv('Future of Jobs AI Dataset.csv')

df.head()

,job_title,country,experience_level,education_level,year,salary,ai_risk_score,primary_skill,skill_demand_score,job_openings,job_survival_class,salary_bucket,ai_risk_category
0,Data Scientist,USA,Senior,Master,2022,6193.103,0.32,Python,88,39158,2,High,Medium Risk
1,Software Engineer,India,Senior,Bachelor,2032,2133.084,0.52,Java,80,7265,1,High,Medium Risk
2,Data Scientist,Canada,Mid,Bachelor,2015,2421.117,0.25,Python,70,22962,1,High,Low Risk
3,Data Scientist,India,Entry,PhD,2034,1179.486,0.44,Python,95,17023,1,Medium,Medium Risk
4,Data Analyst,Canada,Entry,PhD,2035,1799.926,0.75,SQL,61,3433,0,Medium,High Risk


In [9]:
# Features and Targets
X = df[['job_title', 'country', 'experience_level', 'education_level', 'primary_skill']]
y_risk = df['ai_risk_score']
y_salary = df['salary']

In [ ]:
# Preprocessing
nom_features = ['job_title', 'country', 'primary_skill']
ord_features = ['experience_level', 'education_level']

exp_order = ['Entry', 'Mid', 'Senior']
edu_order = ['Bachelor', 'Master', 'PhD']

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', OneHotEncoder(handle_unknown='ignore'), nom_features),
        ('ord', OrdinalEncoder(categories=[exp_order, edu_order]), ord_features)
    ])


In [ ]:
# Pipelines
risk_pipe = Pipeline([
    ('pre', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=100, random_state=42))
])

salary_pipe = Pipeline([
    ('pre', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=100, random_state=42))
])


In [12]:

# Training
X_train, X_test, yr_train, yr_test = train_test_split(X, y_risk, test_size=0.2, random_state=42)
_, _, ys_train, ys_test = train_test_split(X, y_salary, test_size=0.2, random_state=42)

risk_pipe.fit(X_train, yr_train)
salary_pipe.fit(X_train, ys_train)


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('nom',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job_title', 'country',
                                                   'primary_skill']),
                                                 ('ord',
                                                  OrdinalEncoder(categories=[['Entry',
                                                                              'Mid',
                                                                              'Senior'],
                                                                             ['Bachelor',
                                                                              'Master',
                                                                              'PhD']]),
                                                  ['experience_level',
                                                   'education_level'])])),
                ('reg', RandomForestRegressor(random_state=42))])

In [13]:

# Evaluation
print(f"Risk Model R2: {risk_pipe.score(X_test, yr_test):.4f}")
print(f"Salary Model R2: {salary_pipe.score(X_test, ys_test):.4f}")


Risk Model R2: 0.8873
Salary Model R2: 0.8922


In [14]:

# Export
joblib.dump(risk_pipe, 'risk_model.pkl')
joblib.dump(salary_pipe, 'salary_model.pkl')

['salary_model.pkl']